# Soil Classification — Build From Scratch
**MobileNetV2 Transfer Learning · 4 Soil Types · TensorFlow/Keras**

This notebook builds the full pipeline inline — no external `.py` files needed.

**Soil classes:** Alluvial · Black · Red · Yellow

## Cell 1 — Install & Import Dependencies

In [ ]:
# Install all required packages (run once)
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
    'tensorflow', 'numpy', 'matplotlib', 'scikit-learn',
    'pillow', 'seaborn', 'opencv-python'], check=True)

import os
import json
import hashlib
import shutil
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
from collections import defaultdict
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras import layers, regularizers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# CPU thread optimisation — must run before any TF ops
cpu_count = os.cpu_count() or 4
tf.config.threading.set_inter_op_parallelism_threads(cpu_count)
tf.config.threading.set_intra_op_parallelism_threads(cpu_count)

print(f'TensorFlow : {tf.__version__}')
print(f'CPU cores  : {cpu_count}')
print(f'Working dir: {os.getcwd()}')

## Cell 2 — Configuration

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
DATASET_DIR  = 'soil_dataset_4class'    # original images
PREPROCESSED = 'preprocessed_4class'    # CLAHE output
MODEL_PATH   = 'soil_classifier_final.keras'
CLASS_JSON   = 'class_indices.json'

# ── Training ───────────────────────────────────────────────────────────────
BATCH_SIZE    = 32
PHASE1_EPOCHS = 15    # head training (frozen base)
PHASE2_EPOCHS = 15    # fine-tuning (top 50 layers unfrozen)
PHASE1_LR     = 1e-3
PHASE2_LR     = 1e-5
N_UNFREEZE    = 50    # MobileNetV2 layers to unfreeze in Phase 2
USE_FOCAL     = False # True = FocalLoss | False = CrossEntropy + class_weight

# ── ImageNet normalisation (matches MobileNetV2 pre-training) ─────────────
IMAGENET_MEAN = tf.constant([0.485, 0.456, 0.406], dtype=tf.float32)
IMAGENET_STD  = tf.constant([0.229, 0.224, 0.225], dtype=tf.float32)

SOIL_CLASSES  = ['alluvial', 'black', 'red', 'yellow']
MINORITY_CLASSES = set()  # all 4 classes equalised at 480

print('Configuration ready.')
print(f'Dataset    : {DATASET_DIR}')
print(f'Batch size : {BATCH_SIZE}')
print(f'Loss mode  : {"Focal Loss" if USE_FOCAL else "CrossEntropy + class_weight"}')

## Cell 3 — Dataset Audit
Checks for duplicate images (data leakage), low-resolution images, and suspicious mono-colour images.

In [ ]:
EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp'}
MIN_SIZE   = (224, 224)
MONO_THRESH = 0.90

def file_hash(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()

def collect_images(dataset_dir):
    files = []
    for split in ['train', 'validation', 'test']:
        split_dir = os.path.join(dataset_dir, split)
        if not os.path.exists(split_dir):
            continue
        for cls in sorted(os.listdir(split_dir)):
            cls_dir = os.path.join(split_dir, cls)
            if not os.path.isdir(cls_dir):
                continue
            for fname in os.listdir(cls_dir):
                if os.path.splitext(fname)[1].lower() in EXTENSIONS:
                    files.append((split, cls, os.path.join(cls_dir, fname)))
    return files

def audit_dataset(dataset_dir):
    all_files = collect_images(dataset_dir)
    print(f'Total images: {len(all_files)}')

    # 1. Duplicates
    hash_map = defaultdict(list)
    for i, (split, cls, path) in enumerate(all_files):
        hash_map[file_hash(path)].append((split, cls, path))
        if (i+1) % 500 == 0: print(f'  Hashed {i+1}/{len(all_files)}...', end='\r')
    print(f'  Hashed {len(all_files)}/{len(all_files)} done.          ')

    cross_dups, same_dups = [], []
    for h, entries in hash_map.items():
        if len(entries) < 2: continue
        if len({e[0] for e in entries}) > 1:
            cross_dups.append(entries)
        else:
            same_dups.append(entries)

    # 2. Low resolution
    low_res = []
    for _, _, path in all_files:
        try:
            with Image.open(path) as img:
                w, h = img.size
                if w < MIN_SIZE[0] or h < MIN_SIZE[1]:
                    low_res.append(path)
        except: low_res.append(path)

    # 3. Mono-colour
    suspicious = []
    for _, _, path in all_files:
        try:
            arr = np.array(Image.open(path).convert('RGB'), dtype=np.float32)
            dominant = arr.mean(axis=(0,1))
            ratio = (np.abs(arr - dominant).max(axis=2) <= 15).sum() / (arr.shape[0]*arr.shape[1])
            if ratio >= MONO_THRESH:
                suspicious.append(path)
        except: pass

    print(f'Cross-split duplicates (leakage) : {len(cross_dups)}')
    print(f'Same-split duplicates            : {len(same_dups)}')
    print(f'Low-resolution images            : {len(low_res)}')
    print(f'Suspicious mono-colour images    : {len(suspicious)}')
    print(f'Total issues                     : {len(cross_dups)+len(same_dups)+len(low_res)+len(suspicious)}')
    return cross_dups, same_dups, low_res, suspicious

cross_dups, same_dups, low_res, suspicious = audit_dataset(DATASET_DIR)

## Cell 4 — Clean Dataset
Removes all bad images found in the audit. Run once only.

In [ ]:
def clean_dataset(cross_dups, same_dups, low_res, suspicious):
    to_delete = set()

    # Cross-split: keep train copy, remove val/test copies
    priority = {'train': 0, 'validation': 1, 'test': 2}
    for entries in cross_dups:
        entries_sorted = sorted(entries, key=lambda e: priority.get(e[0], 3))
        for entry in entries_sorted[1:]:
            to_delete.add(entry[2])

    # Same-split: keep first, remove rest
    for entries in same_dups:
        for entry in entries[1:]:
            to_delete.add(entry[2])

    # Low-res and suspicious
    for path in low_res + suspicious:
        to_delete.add(path)

    deleted, errors = 0, 0
    for path in to_delete:
        try:
            os.remove(path)
            deleted += 1
        except Exception as e:
            print(f'Error: {e}')
            errors += 1

    print(f'Deleted : {deleted} files')
    print(f'Errors  : {errors}')
    print('Dataset cleaned. Ready for preprocessing.')

clean_dataset(cross_dups, same_dups, low_res, suspicious)

## Cell 5 — Dataset Distribution
Visualise how many images exist per class across train, validation, and test splits.

In [ ]:
CLASS_COLORS = ['#3A7D44', '#4A3728', '#B03A2E', '#C9A227']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

all_counts = {}
for ax, split in zip(axes, ['train', 'validation', 'test']):
    split_dir = os.path.join(DATASET_DIR, split)
    counts = {}
    for cls in sorted(os.listdir(split_dir)):
        cls_dir = os.path.join(split_dir, cls)
        if os.path.isdir(cls_dir):
            counts[cls] = len(os.listdir(cls_dir))
    all_counts[split] = counts

    bars = ax.bar(counts.keys(), counts.values(), color=CLASS_COLORS, edgecolor='white', linewidth=0.8)
    ax.set_title(f'{split.capitalize()} ({sum(counts.values())} images)', fontweight='bold')
    ax.set_ylabel('Images')
    ax.tick_params(axis='x', rotation=20)
    for bar, (cls, n) in zip(bars, counts.items()):
        ax.text(bar.get_x() + bar.get_width()/2, n + 3, str(n),
                ha='center', fontsize=8, fontweight='bold')

plt.suptitle('Class Distribution Across Splits', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150)
plt.show()

print('\nSummary:')
for split, counts in all_counts.items():
    print(f'  {split:12}: {sum(counts.values()):4d} total  |  ', end='')
    print('  '.join(f'{c}={n}' for c, n in counts.items()))

## Cell 6 — Preprocess Dataset (CLAHE)
Applies CLAHE on the L channel of LAB colour space. Removes Gaussian blur.
Run once — output saved to `preprocessed_soil_dataset/`.

In [ ]:
def preprocess_image_clahe(input_path, output_path):
    """
    CLAHE on L channel only (LAB space).
    - Preserves colour (A and B channels untouched)
    - No Gaussian blur — texture must be preserved for soil classification
    """
    img = cv2.imread(input_path)
    if img is None:
        return False
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    img = cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    cv2.imwrite(output_path, img)
    return True

if os.path.exists(PREPROCESSED):
    print(f"'{PREPROCESSED}' already exists — skipping.")
    print("Delete the folder and re-run to reprocess.")
else:
    processed, errors = 0, 0
    for split in ['train', 'validation', 'test']:
        for cls in os.listdir(os.path.join(DATASET_DIR, split)):
            cls_dir = os.path.join(DATASET_DIR, split, cls)
            if not os.path.isdir(cls_dir): continue
            for fname in os.listdir(cls_dir):
                if not fname.lower().endswith(('.jpg','.jpeg','.png','.bmp')): continue
                src = os.path.join(cls_dir, fname)
                dst = os.path.join(PREPROCESSED, split, cls, fname)
                if preprocess_image_clahe(src, dst):
                    processed += 1
                    if processed % 200 == 0: print(f'  {processed} images...', end='\r')
                else:
                    errors += 1
    print(f'\nDone. Processed: {processed}  Errors: {errors}')
    print(f"Output: '{PREPROCESSED}'") 

## Cell 7 — Compare Original vs CLAHE
Side-by-side visual check that preprocessing preserved colour.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))

for col, cls in enumerate(SOIL_CLASSES):
    orig_dir = os.path.join(DATASET_DIR, 'train', cls)
    prep_dir = os.path.join(PREPROCESSED, 'train', cls)
    fname = os.listdir(orig_dir)[0]

    orig = Image.open(os.path.join(orig_dir, fname)).resize((224, 224))
    axes[0][col].imshow(orig)
    axes[0][col].set_title(f'{cls}\nOriginal', fontsize=9)
    axes[0][col].axis('off')

    if os.path.exists(os.path.join(prep_dir, fname)):
        prep = Image.open(os.path.join(prep_dir, fname)).resize((224, 224))
        axes[1][col].imshow(prep)
        axes[1][col].set_title(f'{cls}\nCLAHE', fontsize=9)
    else:
        axes[1][col].set_title('Not preprocessed', fontsize=8, color='red')
    axes[1][col].axis('off')

plt.suptitle('Original vs CLAHE Preprocessed (colour should be preserved)', fontweight='bold')
plt.tight_layout()
plt.show()

## Cell 8 — Focal Loss Definition
Defined inline — no external `losses.py` needed in this notebook.

In [ ]:
class FocalLoss(tf.keras.losses.Loss):
    """
    FL(p_t) = -alpha * (1 - p_t)^gamma * log(p_t)

    gamma=2.0 : down-weights easy examples, focuses on hard/minority classes
    alpha=1.0 : correct scale for multi-class (0.25 is binary detection default)
    """
    def __init__(self, gamma=2.0, alpha=1.0, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.alpha = alpha

    def call(self, y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        p_t = tf.reduce_sum(y_true * y_pred, axis=-1)
        focal_weight = tf.pow(1.0 - p_t, self.gamma)
        return tf.reduce_mean(-self.alpha * focal_weight * tf.math.log(p_t))

    def get_config(self):
        config = super().get_config()
        config.update({'gamma': self.gamma, 'alpha': self.alpha})
        return config

print(f'FocalLoss defined. Mode: {"Focal" if USE_FOCAL else "CrossEntropy"}')
if USE_FOCAL:
    loss_fn = FocalLoss(gamma=2.0, alpha=1.0)
    fit_class_weight = None
    print('  class_weight: DISABLED (Focal handles imbalance)')
else:
    loss_fn = 'categorical_crossentropy'
    print('  class_weight: will be computed in next cell')

## Cell 9 — Class Weights & Data Pipeline
All 4 classes are equalised at 480 images each — equal weights, no oversampling needed.

In [ ]:
# ── Class weights — all 4 classes are equalised at 480 each ──────────────────
train_dir  = os.path.join(PREPROCESSED, 'train')
raw_counts = {
    cls: len(os.listdir(os.path.join(train_dir, cls)))
    for cls in sorted(os.listdir(train_dir))
    if os.path.isdir(os.path.join(train_dir, cls))
}
class_names = sorted(raw_counts.keys())

# Equal weights — dataset is balanced
class_weight_dict = {i: 1.0 for i in range(len(class_names))}
fit_class_weight  = None if USE_FOCAL else class_weight_dict

print('Class counts (equalised):')
for i, (cls, n) in enumerate(zip(class_names, [raw_counts[c] for c in class_names])):
    print(f'  [{i}] {cls:14}: {n:4d} images  w=1.0')

# Save class indices
class_indices = {name: i for i, name in enumerate(class_names)}
with open(CLASS_JSON, 'w') as f:
    json.dump(class_indices, f)
print(f'\nSaved class indices → {CLASS_JSON}')
print(f'Classes: {class_names}')

# steps_per_epoch — simple since no oversampling
total_train = sum(raw_counts.values())
steps_per_epoch = max(1, total_train // BATCH_SIZE)
print(f'\nTotal training samples : {total_train:,}')
print(f'Steps per epoch        : {steps_per_epoch}')

## Cell 10 — Build tf.data Pipeline
Standard augmentation for all 4 balanced classes with ImageNet normalisation.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def build_aug():
    """Standard augmentation for all 4 balanced classes."""
    return tf.keras.Sequential([
        layers.RandomFlip('horizontal_and_vertical'),
        layers.RandomRotation(0.20),
        layers.RandomZoom(0.15),
        layers.RandomTranslation(0.10, 0.10),
        layers.RandomBrightness(0.10),
        layers.RandomContrast(0.10),
    ], name='standard_aug')

def build_datasets(data_dir, batch_size):
    train_ds_raw = tf.keras.utils.image_dataset_from_directory(
        f'{data_dir}/train', label_mode='categorical',
        image_size=(224,224), batch_size=batch_size, shuffle=True, seed=42)
    val_ds = tf.keras.utils.image_dataset_from_directory(
        f'{data_dir}/validation', label_mode='categorical',
        image_size=(224,224), batch_size=batch_size, shuffle=False)
    test_ds = tf.keras.utils.image_dataset_from_directory(
        f'{data_dir}/test', label_mode='categorical',
        image_size=(224,224), batch_size=batch_size, shuffle=False)

    cnames = train_ds_raw.class_names
    aug    = build_aug()

    def preprocess_train(image, label):
        image = tf.cast(image, tf.float32) / 255.0
        image = aug(image, training=True)
        return (image - IMAGENET_MEAN) / IMAGENET_STD, label

    def preprocess_eval(image, label):
        image = tf.cast(image, tf.float32) / 255.0
        return (image - IMAGENET_MEAN) / IMAGENET_STD, label

    train_ds = (train_ds_raw
                .map(preprocess_train, num_parallel_calls=AUTOTUNE)
                .repeat()
                .prefetch(AUTOTUNE))
    val_ds  = val_ds.map(preprocess_eval,  num_parallel_calls=AUTOTUNE).cache().prefetch(AUTOTUNE)
    test_ds = test_ds.map(preprocess_eval, num_parallel_calls=AUTOTUNE).cache().prefetch(AUTOTUNE)
    return train_ds, val_ds, test_ds, cnames

train_ds, val_ds, test_ds, class_names = build_datasets(PREPROCESSED, BATCH_SIZE)

for xb, yb in train_ds.take(1):
    print(f'Batch shape  : {xb.shape}')
    print(f'Label shape  : {yb.shape}')
    print(f'Pixel range  : [{xb.numpy().min():.2f}, {xb.numpy().max():.2f}]  (ImageNet normalised)')
print(f'Classes      : {class_names}')

## Cell 11 — Build MobileNetV2 Model

In [ ]:
def build_model(num_classes=4, input_shape=(224, 224, 3)):
    base = MobileNetV2(weights='imagenet', include_top=False, input_shape=input_shape)
    base.trainable = False  # freeze for Phase 1
    x = base.output
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.BatchNormalization(name='head_bn')(x)
    x = layers.Dense(128, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-4), name='head_dense')(x)
    x = layers.Dropout(0.3, name='head_dropout')(x)
    out = layers.Dense(num_classes, activation='softmax', name='output')(x)
    return Model(inputs=base.input, outputs=out), base

def unfreeze_top_layers(base, n=50):
    base.trainable = True
    for layer in base.layers[:len(base.layers)-n]:
        layer.trainable = False
    # Keep ALL BatchNorm in base frozen — small batches corrupt running stats
    bn_frozen = 0
    for layer in base.layers[len(base.layers)-n:]:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
            bn_frozen += 1
    trainable = sum(tf.size(w).numpy() for w in base.trainable_weights)
    frozen    = sum(tf.size(w).numpy() for w in base.non_trainable_weights)
    print(f'  Unfrozen layers : {sum(1 for l in base.layers if l.trainable)}')
    print(f'  BN kept frozen  : {bn_frozen}')
    print(f'  Trainable params: {trainable:,}')
    print(f'  Frozen params   : {frozen:,}')

model, base_model = build_model(num_classes=len(class_names))

total     = model.count_params()
trainable = sum(tf.size(w).numpy() for w in model.trainable_weights)
print(f'Total parameters    : {total:,}')
print(f'Trainable (Phase 1) : {trainable:,}  (head only)')
print(f'Frozen (base)       : {total - trainable:,}')
print(f'Input shape         : {model.input_shape}')
model.summary(line_length=80)

## Cell 12 — Phase 1 Training (Frozen Base)

In [ ]:
def make_callbacks(ckpt_path, patience=10):
    return [
        EarlyStopping(monitor='val_accuracy', mode='max', patience=patience,
                      restore_best_weights=True, verbose=1, min_delta=1e-3),
        ModelCheckpoint(ckpt_path, monitor='val_accuracy', mode='max',
                        save_best_only=True, verbose=1),
        ReduceLROnPlateau(monitor='val_accuracy', mode='max', factor=0.5,
                          patience=3, min_lr=1e-8, verbose=1),
    ]

model.compile(
    optimizer=Adam(learning_rate=PHASE1_LR, clipnorm=1.0),
    loss=loss_fn,
    metrics=['accuracy']
)

print('=' * 55)
print('PHASE 1 — Head training (MobileNetV2 base frozen)')
print(f'  LR={PHASE1_LR}  |  max_epochs={PHASE1_EPOCHS}')
print('=' * 55)

history_p1 = model.fit(
    train_ds,
    epochs=PHASE1_EPOCHS,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_ds,
    class_weight=fit_class_weight,
    callbacks=make_callbacks('soil_classifier_initial.keras'),
    verbose=1
)

best_p1 = max(history_p1.history['val_accuracy'])
print(f'\nPhase 1 best val_accuracy: {best_p1:.4f}')

## Cell 13 — Phase 2 Fine-Tuning (Top 50 Layers)

In [ ]:
unfreeze_top_layers(base_model, n=N_UNFREEZE)

model.compile(
    optimizer=Adam(learning_rate=PHASE2_LR, clipnorm=1.0),
    loss=loss_fn,
    metrics=['accuracy']
)

print('=' * 55)
print(f'PHASE 2 — Fine-tuning top {N_UNFREEZE} layers')
print(f'  LR={PHASE2_LR}  |  max_epochs={PHASE2_EPOCHS}')
print('=' * 55)

history_p2 = model.fit(
    train_ds,
    epochs=PHASE1_EPOCHS + PHASE2_EPOCHS,
    steps_per_epoch=steps_per_epoch,
    initial_epoch=history_p1.epoch[-1],
    validation_data=val_ds,
    class_weight=fit_class_weight,
    callbacks=make_callbacks('soil_classifier_final.keras', patience=10),
    verbose=1
)

best_p2 = max(history_p2.history['val_accuracy'])
print(f'\nPhase 2 best val_accuracy : {best_p2:.4f}')
print(f'Improvement over Phase 1  : {best_p2 - best_p1:+.4f}')

## Cell 14 — Training History Plot

In [ ]:
acc      = history_p1.history['accuracy']     + history_p2.history['accuracy']
val_acc  = history_p1.history['val_accuracy'] + history_p2.history['val_accuracy']
loss_h   = history_p1.history['loss']         + history_p2.history['loss']
val_loss = history_p1.history['val_loss']     + history_p2.history['val_loss']
p1_end   = len(history_p1.history['accuracy'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for ax, tr, va, title in [
    (ax1, acc,    val_acc,  'Accuracy'),
    (ax2, loss_h, val_loss, 'Loss'),
]:
    ax.plot(tr, label='Train',      color='#3A7D44')
    ax.plot(va, label='Validation', color='#C47C2B')
    ax.axvline(p1_end-1, color='gray', linestyle='--', linewidth=1.2, label='Phase 1→2')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_plots.png', dpi=150)
plt.show()
print('Saved → training_plots.png')

## Cell 15 — Evaluate on Test Set

In [ ]:
# Load best saved model
model = tf.keras.models.load_model(
    'soil_classifier_final.keras',
    custom_objects={'FocalLoss': FocalLoss}
)

preds  = model.predict(test_ds, verbose=1)
y_pred = np.argmax(preds, axis=1)
y_true = []
for _, lbl in test_ds:
    y_true.extend(np.argmax(lbl.numpy(), axis=1))
y_true = np.array(y_true)

loss_val, acc_val = model.evaluate(test_ds, verbose=0)
print(f'\nTest Accuracy : {acc_val*100:.2f}%')
print(f'Test Loss     : {loss_val:.4f}')

## Cell 16 — Classification Report

In [ ]:
print(classification_report(y_true, y_pred, target_names=class_names))

## Cell 17 — Confusion Matrices

In [ ]:
cm      = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
labels  = [c.capitalize() for c in class_names]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, fmt, title, cmap in [
    (ax1, cm,      'd',   'Confusion Matrix (counts)',   'Greens'),
    (ax2, cm_norm, '.2f', 'Normalised Confusion Matrix', 'Blues'),
]:
    sns.heatmap(data, annot=True, fmt=fmt, cmap=cmap,
                xticklabels=labels, yticklabels=labels,
                ax=ax, linewidths=0.5)
    ax.set_xlabel('Predicted', labelpad=8)
    ax.set_ylabel('True',      labelpad=8)
    ax.set_title(title, fontweight='bold')
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

## Cell 18 — Per-class F1 Chart

In [ ]:
from sklearn.metrics import classification_report as cr_dict
report    = cr_dict(y_true, y_pred, target_names=class_names, output_dict=True)
f1_vals   = [report[c]['f1-score'] for c in class_names]
bar_colors = ['#3A7D44','#4A3728','#B03A2E','#C9A227']
macro_f1  = report['macro avg']['f1-score']

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar([c.capitalize() for c in class_names], f1_vals,
              color=bar_colors, edgecolor='white')
for bar, val in zip(bars, f1_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{val*100:.1f}%', ha='center', fontsize=9, fontweight='bold')
ax.axhline(macro_f1, color='gray', linestyle='--', linewidth=1.2,
           label=f'Macro avg {macro_f1*100:.1f}%')
ax.set_ylim(0, 1.15)
ax.set_ylabel('F1 Score')
ax.set_title('Per-class F1 Score', fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('f1_scores.png', dpi=150)
plt.show()

## Cell 19 — Single Image Prediction

In [ ]:
def predict_single(image_path):
    img   = Image.open(image_path).convert('RGB').resize((224, 224))
    arr   = np.array(img, dtype=np.float32) / 255.0
    arr   = (arr - np.array([0.485,0.456,0.406])) / np.array([0.229,0.224,0.225])
    probs = model.predict(np.expand_dims(arr, 0), verbose=0)[0]
    idx   = np.argmax(probs)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    ax1.imshow(Image.open(image_path))
    ax1.set_title(f'Predicted: {class_names[idx].capitalize()}\n'
                  f'Confidence: {probs[idx]*100:.1f}%', fontweight='bold')
    ax1.axis('off')
    colors = ['#2ecc71' if i==idx else '#bdc3c7' for i in range(len(class_names))]
    ax2.barh([c.capitalize() for c in class_names], probs, color=colors)
    ax2.set_xlim(0, 1)
    ax2.set_xlabel('Probability')
    ax2.set_title('Class Probabilities')
    for i, p in enumerate(probs):
        ax2.text(p+0.01, i, f'{p*100:.1f}%', va='center', fontsize=9)
    plt.tight_layout()
    plt.show()
    return class_names[idx], float(probs[idx])

# Replace with any image path from your dataset
sample = os.path.join(DATASET_DIR, 'test', 'alluvial', os.listdir(os.path.join(DATASET_DIR,'test','alluvial'))[0])
print(f'Predicting: {sample}')
predict_single(sample)

## Cell 20 — Grad-CAM Visualisation
Shows which image regions the model focused on. Useful for diagnosing misclassifications.

In [ ]:
import cv2 as cv2_gradcam

def find_last_conv(model):
    for layer in reversed(model.layers):
        if len(layer.output_shape) == 4:
            return layer.name
    raise ValueError('No 4D conv layer found')

def get_gradcam(img_array, model, layer_name):
    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer(layer_name).output, model.output])
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_array)
        pred_idx = tf.argmax(preds[0])
        class_ch = preds[:, pred_idx]
    grads   = tape.gradient(class_ch, conv_out)
    pooled  = tf.reduce_mean(grads, axis=(0,1,2))
    heatmap = conv_out[0] @ pooled[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

def show_gradcam(image_path, model):
    layer  = find_last_conv(model)
    img    = Image.open(image_path).convert('RGB').resize((224,224))
    orig   = np.array(img)
    arr    = (orig.astype(np.float32)/255.0 - np.array([0.485,0.456,0.406])) / np.array([0.229,0.224,0.225])
    inp    = np.expand_dims(arr, 0)
    heatmap = get_gradcam(inp, model, layer)
    heatmap = cv2_gradcam.resize(heatmap, (224,224))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2_gradcam.applyColorMap(heatmap, cv2_gradcam.COLORMAP_JET)
    overlay = np.clip(heatmap * 0.4 + orig, 0, 255).astype('uint8')

    preds   = model.predict(inp, verbose=0)[0]
    pred_cls = class_names[np.argmax(preds)]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    ax1.imshow(orig);     ax1.set_title('Original');                ax1.axis('off')
    ax2.imshow(overlay);  ax2.set_title(f'Grad-CAM → {pred_cls}'); ax2.axis('off')
    plt.suptitle(os.path.basename(image_path), fontsize=10)
    plt.tight_layout()
    plt.show()

# Run on one image per class
for cls in class_names:
    cls_dir = os.path.join(DATASET_DIR, 'test', cls)
    img_path = os.path.join(cls_dir, os.listdir(cls_dir)[0])
    show_gradcam(img_path, model)

## Cell 21 — Launch Streamlit App
Run this command in a **terminal** (not inside Jupyter).

In [ ]:
print('To launch the web UI, run in your terminal:')
print()
print('    streamlit run app.py')
print()
print('Then open http://localhost:8501')